# Cart-pole direct transcription (Euler + SNOPT)

PyDrake URDF cart-pole, decision-variable time step $h$, explicit Euler defect, SNOPT solve, and Meshcat rollout.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    DiagramBuilder,
    LogVectorOutput,
    MathematicalProgram,
    Parser,
    PiecewisePolynomial,
    Simulator,
    SnoptSolver,
    StartMeshcat,
    TrajectorySource,
)
from pydrake.visualization import AddDefaultVisualization
from underactuated import ConfigureParser

In [ ]:
def make_cartpole():
    builder = DiagramBuilder()
    plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
    parser = Parser(plant)
    ConfigureParser(parser)
    parser.AddModelsFromUrl("package://underactuated/models/undamped_cartpole.urdf")
    plant.Finalize()
    return builder.Build(), plant


diagram, cartpole = make_cartpole()
dyn_context = diagram.CreateDefaultContext()
dyn_plant_context = cartpole.GetMyMutableContextFromRoot(dyn_context)

x_initial = np.array([0.0, 0.0, 0.0, 0.0])
x_target = np.array([0.0, np.pi, 0.0, 0.0])

def f_cartpole(x, u):
    dyn_plant_context.get_mutable_continuous_state_vector().SetFromVector(np.asarray(x, dtype=float))
    cartpole.get_actuation_input_port().FixValue(dyn_plant_context, np.asarray(u, dtype=float))
    return cartpole.EvalTimeDerivatives(dyn_plant_context).CopyToVector()

Direct transcription:

\[
    x_{k+1} - x_k - h f(x_k,u_k) = 0.
\]

In [ ]:
N = 151
nX, nU = 4, 1

h_min, h_max, h_guess = 0.02, 0.06, 0.03
x_min = np.array([-10.0, -10.0, -10.0, -10.0])
x_max = np.array([ 10.0,  10.0,  10.0,  10.0])
u_min = np.array([-100.0])
u_max = np.array([ 100.0])
terminal_tol = np.array([1e-3, 1e-3, 5e-3, 5e-3])

Q = np.diag([0.01, 0.01, 0.01, 0.01])
Qf = np.diag([10.0, 200.0, 10.0, 10.0])
R = np.diag([0.01])
time_weight = 0.1

def qform(z, Q):
    return sum(Q[i, j] * z[i] * z[j] for i in range(Q.shape[0]) for j in range(Q.shape[1]))

def euler_defect(z):
    xk = z[:4]
    uk = z[4:5]
    xkp1 = z[5:9]
    hk = z[9]
    return xkp1 - xk - hk * f_cartpole(xk, uk)

In [ ]:
prog = MathematicalProgram()
x = prog.NewContinuousVariables(nX, N, "x")
u = prog.NewContinuousVariables(nU, N - 1, "u")
h = prog.NewContinuousVariables(1, "h")[0]

prog.AddBoundingBoxConstraint(x_initial, x_initial, x[:, 0])
prog.AddBoundingBoxConstraint(x_target - terminal_tol, x_target + terminal_tol, x[:, -1])
prog.AddBoundingBoxConstraint(h_min, h_max, np.array([h]))

for k in range(N):
    prog.AddBoundingBoxConstraint(x_min, x_max, x[:, k])
for k in range(N - 1):
    prog.AddBoundingBoxConstraint(u_min, u_max, u[:, k])
    prog.AddConstraint(euler_defect, np.zeros(nX), np.zeros(nX), np.r_[x[:, k], u[:, k], x[:, k + 1], h])
    prog.AddCost(h * (qform(x[:, k] - x_target, Q) + qform(u[:, k], R)))

prog.AddCost(qform(x[:, -1] - x_target, Qf))
prog.AddCost(time_weight * (N - 1) * h)

prog.SetSolverOption(SnoptSolver.id(), "Derivative Option", 0)
prog.SetSolverOption(SnoptSolver.id(), "Major Feasibility Tolerance", 1e-8)
prog.SetSolverOption(SnoptSolver.id(), "Major Optimality Tolerance", 1e-8)
prog.SetSolverOption(SnoptSolver.id(), "Minor Feasibility Tolerance", 1e-8)
prog.SetSolverOption(SnoptSolver.id(), "Iterations Limit", 10000)
prog.SetSolverOption(SnoptSolver.id(), "Major Iterations Limit", 10000)

for i in range(nX):
    prog.SetInitialGuess(x[i, :], np.linspace(x_initial[i], x_target[i], N))
prog.SetInitialGuess(u, np.zeros((nU, N - 1)))
prog.SetInitialGuess(h, h_guess)

In [ ]:
result = SnoptSolver().Solve(prog)

x_sol = result.GetSolution(x)
u_sol = result.GetSolution(u)
h_sol = result.GetSolution(h)
tx = np.arange(N) * h_sol
tu = np.arange(N - 1) * h_sol

print("solver used:", result.get_solver_id().name())
print("success:", result.is_success())
print("solution result:", result.get_solution_result())
print("SNOPT info:", result.get_solver_details().info)
print("optimal h =", h_sol)
print("total time =", (N - 1) * h_sol)
print("transcription final state =", x_sol[:, -1])
print("terminal error =", x_sol[:, -1] - x_target)
print("terminal error inf-norm =", np.linalg.norm(x_sol[:, -1] - x_target, ord=np.inf))

D = np.column_stack([
    x_sol[:, k + 1] - x_sol[:, k] - h_sol * f_cartpole(x_sol[:, k], u_sol[:, k])
    for k in range(N - 1)
])
print("max abs Euler defect =", np.max(np.abs(D)))

In [ ]:
plt.figure()
plt.plot(x_sol[0, :], x_sol[1, :], "-o", markersize=2)
plt.plot(x_initial[0], x_initial[1], "go", label="start")
plt.plot(x_target[0], x_target[1], "ro", label="target")
plt.xlabel("cart position")
plt.ylabel("pole angle")
plt.title("Cart-pole swing-up trajectory")
plt.grid(True)
plt.legend()

plt.figure()
plt.plot(tx, x_sol[0, :])
plt.xlabel("time [s]")
plt.ylabel("cart position")
plt.grid(True)

plt.figure()
plt.plot(tx, x_sol[1, :])
plt.xlabel("time [s]")
plt.ylabel("pole angle")
plt.grid(True)

plt.figure()
plt.step(tu, u_sol[0, :], where="post")
plt.xlabel("time [s]")
plt.ylabel("cart force")
plt.grid(True)

plt.figure()
plt.plot(x_sol[1, :], x_sol[3, :])
plt.xlabel("pole angle")
plt.ylabel("angular velocity")
plt.title("Angle phase portrait")
plt.grid(True)
plt.show()

Roll out the optimized zero-order-hold input in Meshcat.

In [ ]:
meshcat = StartMeshcat()
meshcat.Delete()

rollout_times = np.arange(N) * h_sol
u_samples = np.hstack([u_sol, u_sol[:, -1:]])
u_traj = PiecewisePolynomial.ZeroOrderHold(rollout_times, u_samples)

builder = DiagramBuilder()
cartpole_sim, scene_graph_sim = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
parser = Parser(cartpole_sim)
ConfigureParser(parser)
parser.AddModelsFromUrl("package://underactuated/models/undamped_cartpole.urdf")
cartpole_sim.Finalize()

u_source = builder.AddSystem(TrajectorySource(u_traj))
builder.Connect(u_source.get_output_port(), cartpole_sim.get_actuation_input_port())
state_logger = LogVectorOutput(cartpole_sim.get_state_output_port(), builder)
AddDefaultVisualization(builder, meshcat)

sim_diagram = builder.Build()
sim_context = sim_diagram.CreateDefaultContext()
sim_plant_context = cartpole_sim.GetMyMutableContextFromRoot(sim_context)
sim_plant_context.get_mutable_continuous_state_vector().SetFromVector(x_initial)

simulator = Simulator(sim_diagram, sim_context)
simulator.get_mutable_integrator().set_maximum_step_size(min(h_sol / 20.0, 0.002))
simulator.set_target_realtime_rate(1.0)

meshcat.StartRecording()
simulator.Initialize()
simulator.AdvanceTo(rollout_times[-1])
meshcat.PublishRecording()

log = state_logger.FindLog(sim_context)
t_rollout = log.sample_times()
x_rollout = log.data()

print("rollout final state =", x_rollout[:, -1])
print("rollout terminal error =", x_rollout[:, -1] - x_target)
print("rollout terminal error inf-norm =", np.linalg.norm(x_rollout[:, -1] - x_target, ord=np.inf))

In [ ]:
plt.figure()
plt.plot(tx, x_sol[0, :], "o-", markersize=2, label="NLP knot x")
plt.plot(t_rollout, x_rollout[0, :], label="rollout x")
plt.xlabel("time [s]")
plt.ylabel("cart position")
plt.grid(True)
plt.legend()

plt.figure()
plt.plot(tx, x_sol[1, :], "o-", markersize=2, label="NLP knot theta")
plt.plot(t_rollout, x_rollout[1, :], label="rollout theta")
plt.xlabel("time [s]")
plt.ylabel("pole angle")
plt.grid(True)
plt.legend()

plt.figure()
plt.plot(x_sol[0, :], x_sol[1, :], "o-", markersize=2, label="NLP knots")
plt.plot(x_rollout[0, :], x_rollout[1, :], label="rollout")
plt.xlabel("cart position")
plt.ylabel("pole angle")
plt.grid(True)
plt.legend()
plt.show()